# 08b — Balance Dataset & Run AutoML

Downloads all images from GCS buckets, augments types 4/5/6 to balance the dataset, uploads to a project bucket, and runs Vertex AI AutoML.

**Data sources:**
- Fitzpatrick17k → `gs://fitzpatrick-dataset/all_images/`
- SCIN → `gs://dx-scin-public-data/dataset/images/`

**Augmentation (Albumentations):** flips, rotations, random crop, Gaussian noise
**Forbidden:** brightness, contrast, color jitter, blurring

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
!pip install -q albumentations

sys.path.insert(0, "/content/NST_Class")
print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

In [ ]:
# Cell 3: Config & load CSV
import pandas as pd
import numpy as np
from pathlib import Path

# --- GCS buckets ---
FITZ_BUCKET = "gs://fitzpatrick-dataset/all_images"      # friend's bucket with all Fitz17k images
SCIN_BUCKET = "gs://dx-scin-public-data/dataset/images"  # public SCIN bucket
PROJECT_BUCKET = "skin-tone-project"                      # our project bucket for output
GCS_IMAGE_PREFIX = f"gs://{PROJECT_BUCKET}/balanced_images"

# --- Local paths ---
LOCAL_IMAGE_DIR = "data/combined_images"
LOCAL_AUG_DIR = "data/augmented_images"

# --- Augmentation ---
SEED = 42
IMAGE_SIZE = 224
TARGET_PER_CLASS = 2500  # all minority classes augmented to this count

df = pd.read_csv("combined_dataset.csv")
print("Combined dataset:")
print(df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(df)}")
print(f"\nBy source:")
print(df["source"].value_counts())

# Compute augmentation targets dynamically — bring types 4/5/6 up to TARGET_PER_CLASS
print(f"\nAugmentation plan (target {TARGET_PER_CLASS} per class):")
AUGMENTATION_TARGETS = {}
for cls in [4, 5, 6]:
    current = len(df[df["fitzpatrick_scale"] == cls])
    needed = max(0, TARGET_PER_CLASS - current)
    if needed > 0:
        AUGMENTATION_TARGETS[cls] = needed
    print(f"  Type {cls}: {current} + {needed} augmented = {current + needed}")

In [ ]:
# Cell 4a: Download Fitzpatrick17k from GCS bucket
import tempfile, time

os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)

fitz_ids = set(df[df["source"] == "fitzpatrick17k"]["img_id"])
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()} if os.path.isdir(LOCAL_IMAGE_DIR) else set()
needed_fitz = fitz_ids - existing

if not needed_fitz:
    print(f"Fitzpatrick17k: all {len(fitz_ids)} images already present")
else:
    print(f"Fitzpatrick17k: downloading {len(needed_fitz)} images from {FITZ_BUCKET}...")
    manifest_path = os.path.join(tempfile.gettempdir(), "fitz_manifest.txt")
    with open(manifest_path, "w") as f:
        for img_id in sorted(needed_fitz):
            f.write(f"{FITZ_BUCKET}/{img_id}\n")

    os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "16"
    os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"

    start = time.time()
    !cat {manifest_path} | gcloud storage cp -I "{LOCAL_IMAGE_DIR}/"
    elapsed = time.time() - start

    new_existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
    downloaded = len((new_existing & fitz_ids) - existing)
    print(f"\nFitzpatrick17k: {downloaded}/{len(needed_fitz)} downloaded in {elapsed:.0f}s")

In [ ]:
# Cell 4b: Download SCIN from public GCS bucket
scin_ids = set(df[df["source"] == "scin"]["img_id"])
existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
needed_scin = scin_ids - existing

if not needed_scin:
    print(f"SCIN: all {len(scin_ids)} images already present")
else:
    print(f"SCIN: downloading {len(needed_scin)} images from {SCIN_BUCKET}...")
    manifest_path = os.path.join(tempfile.gettempdir(), "scin_manifest.txt")
    with open(manifest_path, "w") as f:
        for img_id in sorted(needed_scin):
            f.write(f"{SCIN_BUCKET}/{img_id}\n")

    start = time.time()
    !cat {manifest_path} | gcloud storage cp -I "{LOCAL_IMAGE_DIR}/"
    elapsed = time.time() - start

    new_existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}
    downloaded = len((new_existing & scin_ids) - existing)
    print(f"\nSCIN: {downloaded}/{len(needed_scin)} downloaded in {elapsed:.0f}s")

In [ ]:
# Cell 4c: Verify downloads
from PIL import Image as PILImage
from tqdm import tqdm

existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}

# Only count Fitz + SCIN (PAD-UFES excluded from this notebook)
fitz_scin_df = df[df["source"].isin(["fitzpatrick17k", "scin"])].copy()
available_ids = set()
corrupt = 0

for img_id in tqdm(fitz_scin_df["img_id"], desc="Validating"):
    if img_id not in existing:
        continue
    try:
        with PILImage.open(os.path.join(LOCAL_IMAGE_DIR, img_id)) as img:
            img.verify()
        available_ids.add(img_id)
    except Exception:
        corrupt += 1

df_available = fitz_scin_df[fitz_scin_df["img_id"].isin(available_ids)].copy()

print(f"\nValidation:")
print(f"  Fitzpatrick17k: {len(df_available[df_available['source'] == 'fitzpatrick17k'])}/{len(df[df['source'] == 'fitzpatrick17k'])}")
print(f"  SCIN:           {len(df_available[df_available['source'] == 'scin'])}/{len(df[df['source'] == 'scin'])}")
print(f"  Corrupt:        {corrupt}")
print(f"  Total usable:   {len(df_available)}")
print(f"\nPer-class (usable):")
for cls in sorted(df_available["fitzpatrick_scale"].unique()):
    print(f"  Type {cls}: {len(df_available[df_available['fitzpatrick_scale'] == cls])}")

In [ ]:
# Cell 5: Augment minority classes (types 4, 5, 6)
from scripts.augment_minority import augment_images, verify_augmentation

all_augmented = []

for fitz_class, target_count in sorted(AUGMENTATION_TARGETS.items()):
    print(f"{'='*50}")
    print(f"Augmenting Fitzpatrick type {fitz_class}")
    print(f"{'='*50}")

    class_imgs = df_available[df_available["fitzpatrick_scale"] == fitz_class]["img_id"].tolist()
    print(f"Source images available: {len(class_imgs)}")
    print(f"Target new images: {target_count}")

    if not class_imgs:
        print(f"  SKIPPED — no source images available for type {fitz_class}")
        continue

    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    created, paths = augment_images(
        input_dir=LOCAL_IMAGE_DIR,
        output_dir=class_aug_dir,
        img_ids=class_imgs,
        target_count=target_count,
        seed=SEED + fitz_class,
        image_size=IMAGE_SIZE,
    )

    print(f"Created: {created} augmented images")

    result = verify_augmentation(LOCAL_IMAGE_DIR, class_aug_dir)
    print(f"Verification: {result['valid']}/{result['checked']} valid (passed={result['passed']})")
    assert result["passed"], f"Verification failed for type {fitz_class}: {result}"

    all_augmented.extend([
        {"img_id": p.name, "fitzpatrick_scale": fitz_class, "source": "augmented"}
        for p in paths
    ])

print(f"\nTotal augmented: {len(all_augmented)}")

In [ ]:
# Cell 6: Visual spot-check
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(AUGMENTATION_TARGETS), 3, figsize=(12, 4 * len(AUGMENTATION_TARGETS)))

for row, fitz_class in enumerate(sorted(AUGMENTATION_TARGETS.keys())):
    class_aug_dir = Path(f"{LOCAL_AUG_DIR}/type_{fitz_class}")
    aug_files = sorted(class_aug_dir.glob("*"))[:3]
    for col, img_path in enumerate(aug_files):
        ax = axes[row][col]
        ax.imshow(PILImage.open(img_path))
        ax.set_title(f"Type {fitz_class}: {img_path.name}", fontsize=8)
        ax.axis("off")

plt.suptitle("Augmented Image Spot-Check (verify no artifacts)", fontsize=14)
plt.tight_layout()
plt.show()
print("Visually inspect the images above.")

In [ ]:
# Cell 7: Build balanced CSV
aug_df = pd.DataFrame(all_augmented)
balanced_df = pd.concat([df_available, aug_df], ignore_index=True)

print("Balanced dataset:")
print(balanced_df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(balanced_df)}")
print(f"  Original (available): {len(df_available)}")
print(f"  Augmented: {len(aug_df)}")

balanced_df.to_csv("balanced_dataset.csv", index=False)
print("\nSaved to balanced_dataset.csv")

In [ ]:
# Cell 8: Upload ALL images (originals + augmented) to project bucket
os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "16"
os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"

print("Uploading original images...")
!gcloud storage cp -n "{LOCAL_IMAGE_DIR}/*" "{GCS_IMAGE_PREFIX}/"

print("\nUploading augmented images...")
for fitz_class in sorted(AUGMENTATION_TARGETS.keys()):
    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    if os.path.isdir(class_aug_dir):
        !gcloud storage cp -r "{class_aug_dir}/*" "{GCS_IMAGE_PREFIX}/"

!gcloud storage cp balanced_dataset.csv "gs://{PROJECT_BUCKET}/balanced_dataset.csv"
print("\nAll uploads complete!")

In [ ]:
# Cell 9: Generate AutoML manifest with stratified split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    balanced_df, test_size=0.30, random_state=SEED,
    stratify=balanced_df["fitzpatrick_scale"],
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["fitzpatrick_scale"],
)

train_df = train_df.copy(); train_df["split"] = "TRAINING"
val_df = val_df.copy(); val_df["split"] = "VALIDATION"
test_df = test_df.copy(); test_df["split"] = "TEST"

manifest_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

rows = []
for _, row in manifest_df.iterrows():
    gcs_path = f"{GCS_IMAGE_PREFIX}/{row['img_id']}"
    rows.append(f"{row['split']},{gcs_path},{int(row['fitzpatrick_scale'])}")

manifest_path = "data/automl_balanced_manifest.csv"
os.makedirs("data", exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rows) + "\n")

print(f"Manifest: {manifest_path} ({len(rows)} rows)")
print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nPer-class in train:")
print(train_df["fitzpatrick_scale"].value_counts().sort_index())

!gcloud storage cp {manifest_path} "gs://{PROJECT_BUCKET}/automl/balanced_manifest.csv"
print("\nManifest uploaded!")

In [ ]:
# Cell 10: Run AutoML
from google.cloud import aiplatform

PROJECT_ID = "project-9b17a620-5dfc-40d7-84a"
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

dataset = aiplatform.ImageDataset.create(
    display_name="skin-tone-balanced",
    gcs_source=f"gs://{PROJECT_BUCKET}/automl/balanced_manifest.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.single_label_classification,
)
print(f"Dataset created: {dataset.resource_name}")

job = aiplatform.AutoMLImageTrainingJob(
    display_name="skin-tone-balanced-automl",
    prediction_type="classification",
    multi_label=False,
    model_type="CLOUD",
)

model = job.run(
    dataset=dataset,
    model_display_name="skin-tone-balanced-v1",
    budget_milli_node_hours=8000,
)
print(f"Model trained: {model.resource_name}")

In [ ]:
# Cell 11: Evaluate
model_eval = model.list_model_evaluations()[0]
metrics = model_eval.metrics

print("AutoML Evaluation (Balanced Dataset):")
for key, value in metrics.items():
    print(f"  {key}: {value}")

if "confusionMatrix" in metrics:
    print("\nConfusion Matrix:")
    print(metrics["confusionMatrix"])

print(f"\nModel ID: {model.resource_name}")

In [ ]:
# Cell 12: Summary
print("=" * 60)
print("BALANCED AUGMENTATION SUMMARY")
print("=" * 60)
print(f"\nSources: Fitzpatrick17k ({FITZ_BUCKET}) + SCIN ({SCIN_BUCKET})")
print(f"Available: {len(df_available)} images")
print(f"Augmented: {len(aug_df)}")
print(f"Balanced total: {len(balanced_df)}")
print(f"\nPer-class distribution:")
for cls in range(1, 7):
    orig = len(df_available[df_available["fitzpatrick_scale"] == cls])
    total = len(balanced_df[balanced_df["fitzpatrick_scale"] == cls])
    aug = total - orig
    print(f"  Type {cls}: {orig} original + {aug} augmented = {total}")
print(f"\nProject bucket: gs://{PROJECT_BUCKET}/")
print(f"Images: {GCS_IMAGE_PREFIX}/")
print("=" * 60)